First, upload MappingMovielens2DBpedia-1.2.tsv and ML1M Movie Titles.xlsx

In [11]:
import pandas as pd

# Replace 'ML1M Movie Titles.xlsx' with the actual path to your Excel file.
file_path = 'ML1M Movie Titles.xlsx'

# Read the Excel file into a Pandas DataFrame
df = pd.read_excel(file_path)

# Print the first few rows of the DataFrame to verify that it's loaded correctly
print(df.head())

                          Movie Title
0                    Toy Story (1995)
1                      Jumanji (1995)
2             Grumpier Old Men (1995)
3            Waiting to Exhale (1995)
4  Father of the Bride Part II (1995)


In [6]:
pip install SPARQLWrapper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.8 MB/s eta 0:00:00


In [15]:
pip install tqdm

TESTING with DBpedia composer

In [10]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from multiprocessing import Pool
import time

def get_movie_song_name(movie_data):
    movie_title, dbpedia_uri = movie_data
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?songName
        WHERE {{
            <{dbpedia_uri}> dbo:musicComposer ?composer .
            ?composer rdfs:label ?songName .
            FILTER (LANG(?songName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        song_names = [binding['songName']['value'] for binding in bindings]
        return movie_title, ', '.join(song_names)
    else:
        return movie_title, None

# Replace 'ML1M Movie Titles.xlsx' with the actual path to your Excel file.
file_path = 'ML1M Movie Titles.xlsx'

# Load the Excel file into a Pandas DataFrame
df = pd.read_excel(file_path)

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create new columns for movie song names and DBpedia URI
df['Movie Song Names'] = ""
df['DBpedia URI'] = ""

# Define the number of worker processes to use
num_processes = 4

# Create a pool of worker processes
pool = Pool(processes=num_processes)

# Enrich the DataFrame with movie song names and DBpedia URIs
enriched_results = []
for idx, row in df.iterrows():
    movie_title = row['Movie Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Add the movie data to the list for processing
        enriched_results.append((movie_title, dbpedia_uri))

# Use parallel processing to retrieve song names
song_name_results = pool.map(get_movie_song_name, enriched_results)

# Update the DataFrame with the retrieved information
for movie_title, song_names in song_name_results:
    if song_names is not None:
        df.loc[df['Movie Title'] == movie_title, 'Movie Song Names'] = song_names

    # Update the DBpedia URI in the DataFrame
    df.loc[df['Movie Title'] == movie_title, 'DBpedia URI'] = dbpedia_uri

# Save the enriched DataFrame back to the same Excel file
df.to_excel(file_path, index=False)

# Close the pool of worker processes
pool.close()
pool.join()

FINAL CODE

In [16]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from multiprocessing import Pool
from tqdm import tqdm  # Import tqdm for progress bars
import time

def get_movie_song_name(movie_data):
    movie_title, dbpedia_uri = movie_data
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?songName
        WHERE {{
            <{dbpedia_uri}> dbo:musicComposer ?composer .
            ?composer rdfs:label ?songName .
            FILTER (LANG(?songName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        song_names = [binding['songName']['value'] for binding in bindings]
        return movie_title, ', '.join(song_names)
    else:
        return movie_title, None

# Replace 'ML1M Movie Titles.xlsx' with the actual path to your Excel file.
file_path = 'ML1M Movie Titles.xlsx'

# Load the Excel file into a Pandas DataFrame
df = pd.read_excel(file_path)

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create new columns for Music Composer (formerly "Composer") and DBpedia URI
df['Music Composer'] = ""
df['DBpedia URI'] = ""

# Define the number of worker processes to use
num_processes = 4

# Create a pool of worker processes
pool = Pool(processes=num_processes)

# Enrich the DataFrame with Music Composer and DBpedia URIs
enriched_results = []
start_time = time.time()  # Initialize start_time here
for idx, row in df.iterrows():
    movie_title = row['Movie Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Add the movie data to the list for processing
        enriched_results.append((movie_title, dbpedia_uri))

# Use tqdm to create a progress bar
with tqdm(total=len(enriched_results)) as pbar:
    # Use parallel processing to retrieve Music Composer
    composer_results = pool.map(get_movie_song_name, enriched_results)

    # Update the DataFrame with the retrieved information
    for movie_title, composer_names in composer_results:
        if composer_names is not None:
            df.loc[df['Movie Title'] == movie_title, 'Music Composer'] = composer_names  # Change the column name here

        # Update the DBpedia URI in the DataFrame
        df.loc[df['Movie Title'] == movie_title, 'DBpedia URI'] = dbpedia_uri

        # Update the progress bar
        pbar.update(1)

# Calculate elapsed time
elapsed_time = time.time() - start_time
print(f"Processed {len(enriched_results)} rows. Elapsed time: {elapsed_time:.2f} seconds")

# Save the enriched DataFrame to a new Excel file named "enriched_data.xlsx"
output_file = 'enriched_data.xlsx'
df.to_excel(output_file, index=False)
print(f"Enriched data saved to {output_file}")

# Close the pool of worker processes
pool.close()
pool.join()

100%|██████████| 3227/3227 [00:53<00:00, 60.49it/s] 


Processed 3227 rows. Elapsed time: 56.09 seconds
Enriched data saved to enriched_data.xlsx
